# Weighted 2PCF convergence from GALFORM

> The attached reference image is a 1D weighted $\xi(r)$ convergence plot at $z=1.50$.
This notebook reproduces that comparison using GALFORM subvolumes and the repo's weighted-`xi` helper, which is the correct match for the image.

> Local SCOPE at `/cosma/apps/durham/dc-hick2/SCOPE/scope` computes $\xi(r_p,\pi)$ and $w_p(r_p)$, so it is not the same observable as the reference figure.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from galform_analysis.analysis.correlation.subvol_weighted_correction import compute_weighted_xi_for_n_list

PROJECT_ROOT = Path('/cosma/apps/durham/dc-hick2/galform_analysis')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('Project root:', PROJECT_ROOT)

In [ ]:
# GALFORM data parameters matching the attached figure
BASE_DIR = Path('/cosma5/data/durham/dc-hick2/Galform_Out/L800/lc16')
IZ = 155  # z = 1.50
K_TOTAL = 1024
BOX_SIZE = 542.16  # L800 box size in Mpc/h

# Convergence sweep shown in the reference image.
N_SUBVOLS_LIST = [1, 2, 4, 8, 16, 25, 50, 100, 200, 400, 700, 1024]

# Use all galaxies to match the reference scale; the helper handles subvolume weighting.
CENTRALS_ONLY = False
MHALO_MIN = None
MSTAR_MIN_LOG10 = None

RBINS = np.logspace(-1.0, 1.5, 21)

print('Base dir:', BASE_DIR)
print('Snapshot:', f'iz{IZ}')
print('Box size:', BOX_SIZE)
print('n_subvols sweep:', N_SUBVOLS_LIST)

In [ ]:
# Prefer loading precomputed CSV results rather than running heavy jobs here
import glob
import pandas as pd
from pathlib import Path

out_dir = Path('data/subvolume_weighted_xi')
csvs = sorted(out_dir.glob(f'weighted_xi_iz{IZ}_n*.csv'))
if csvs:
    df = pd.concat([pd.read_csv(p) for p in csvs], ignore_index=True)
    weighted_xi = df.sort_values(['n_subvol', 'bin_idx']).reset_index(drop=True)
    print(f'Loaded {len(csvs)} CSV files, rows={len(weighted_xi)}')
else:
    print('No CSV results found in', out_dir)
    print('Submit batch jobs with:')
    print('  python scripts/subvolume/submit_weighted_xi_jobs.py --base-dir', BASE_DIR, '--iz', IZ, '--n-list', ' '.join(map(str, N_SUBVOLS_LIST)))
    weighted_xi = pd.DataFrame()


In [ ]:
ref = weighted_xi[weighted_xi['n_subvol'] == K_TOTAL][['bin_idx', 'r', 'xi_corrected']].rename(columns={'xi_corrected': 'xi_ref'})
plot_df = weighted_xi.merge(ref, on=['bin_idx', 'r'], how='left')
plot_df['frac_diff'] = 100.0 * (plot_df['xi_corrected'] / plot_df['xi_ref'] - 1.0)

# Keep the same n-values visible in the reference plot ordering.
plot_df['n_subvol'] = pd.Categorical(plot_df['n_subvol'], categories=N_SUBVOLS_LIST, ordered=True)

print(plot_df[['n_subvol', 'r', 'xi_corrected', 'xi_ref', 'frac_diff']].head(12).to_string(index=False))

In [ ]:
summary = (
    plot_df.groupby('n_subvol', observed=False)['frac_diff']
    .agg(median_abs_frac=lambda s: float(np.nanmedian(np.abs(s))))
    .reset_index()256  
    .sort_values('n_subvol')
 )

print(summary.to_string(index=False))
best = summary.loc[summary['median_abs_frac'].idxmin()]
print(f"Best-matching n_subvol by median |fractional difference|: {int(best['n_subvol'])}")

In [ ]:
fig, (ax_top, ax_bottom) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

plot_order = [1024, 1, 2, 4, 8, 16, 25, 50, 100, 200, 400, 700]
plot_order = [n for n in plot_order if n in set(plot_df['n_subvol'].astype(int))]

cmap = plt.get_cmap('viridis', max(2, len(plot_order)))

for idx, n in enumerate(plot_order):
    sub = plot_df[plot_df['n_subvol'].astype(int) == n].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan).dropna(subset=['r', 'xi_corrected'])
    if sub.empty:
        continue
    color = cmap(idx)
    label = f'n={n}' if n != 1024 else 'n=1024 (reference)'
    ax_top.plot(sub['r'], sub['xi_corrected'], marker='o', ms=3, lw=1.5, color=color, label=label)

    if n != 1024:
        ax_bottom.plot(sub['r'], sub['frac_diff'], marker='o', ms=3, lw=1.2, color=color, label=f'n={n}')

ax_top.set_xscale('log')
ax_top.set_yscale('log')
ax_top.set_ylabel(r'$\xi(r)$')
ax_top.set_title(f'Weighted 2PCF convergence at z={1.50:.2f}')
ax_top.grid(True, which='both', alpha=0.25)
ax_top.legend(ncol=2, fontsize=8, frameon=False, loc='upper right')

ax_bottom.axhline(0.0, color='black', ls='--', lw=1)
ax_bottom.set_xscale('log')
ax_bottom.set_xlabel(r'$r\ [h^{-1}\,\mathrm{Mpc}]$')
ax_bottom.set_ylabel(r'$100\,(\xi/\xi_{\rm ref}-1)$')
ax_bottom.grid(True, which='both', alpha=0.25)
ax_bottom.legend(ncol=2, fontsize=8, frameon=False, loc='lower left')

plt.tight_layout()
plt.show()